# Notebook 02: Data Preparation and Fine-Tuning

Prepare training data from our e-commerce FAQ, generate synthetic conversations, and run our first QLoRA fine-tuning.

---
## Part A: Data Preparation
---

## 1. Setup

In [ ]:
import os
import json
import random
import time
import pandas as pd
import matplotlib.pyplot as plt
import torch
from pathlib import Path
from dotenv import load_dotenv
from openai import OpenAI
from datasets import Dataset
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, TaskType
from trl import SFTTrainer, SFTConfig

load_dotenv()

LLM_BASE_URL = os.getenv('LLM_BASE_URL')
LLM_API_KEY = os.getenv('LLM_API_KEY')
LLM_MODEL = os.getenv('LLM_MODEL', 'gpt-4o')
HF_TOKEN = os.getenv('HF_TOKEN')

llm_client = OpenAI(base_url=LLM_BASE_URL, api_key=LLM_API_KEY)

MODEL_ID = 'meta-llama/Llama-3.2-1B-Instruct'
# MODEL_ID = 'microsoft/Phi-3-mini-4k-instruct'  # Fallback (ungated)

# Load the Part 4 FAQ data
faq_path = Path('../../part4/data/ecommerce_faq.csv')
faq_df = pd.read_csv(faq_path)

print(f'[OK] Configuration loaded')
print(f'  LLM Model: {LLM_MODEL}')
print(f'  Fine-tune Model: {MODEL_ID}')
print(f'  FAQ data: {len(faq_df)} Q&A pairs')
print(f'\nSample:')
print(f'  Q: {faq_df.iloc[0]["question"]}')
print(f'  A: {faq_df.iloc[0]["answer"][:80]}...')

## 2. Chat Template Format

LLMs are trained on specific chat formats. When we fine-tune, we need to match the model's expected format.

For Llama 3.2, the format looks like:
```
<|begin_of_text|><|start_header_id|>system<|end_header_id|>

You are a helpful assistant.<|eot_id|>
<|start_header_id|>user<|end_header_id|>

What is your return policy?<|eot_id|>
<|start_header_id|>assistant<|end_header_id|>

Our return policy allows...<|eot_id|>
```

The good news: the tokenizer handles this automatically with `apply_chat_template()`.

In [ ]:
# Load tokenizer to see the chat template in action
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, token=HF_TOKEN)

# Example: single Q&A formatted as chat
example_messages = [
    {'role': 'system', 'content': 'You are a helpful e-commerce customer service agent.'},
    {'role': 'user', 'content': 'What is your return policy?'},
    {'role': 'assistant', 'content': 'Our return policy allows you to return products within 30 days of purchase for a full refund.'}
]

formatted = tokenizer.apply_chat_template(example_messages, tokenize=False)
print('Formatted chat template:')
print(formatted)

## 3. Convert FAQ Data to Instruction Format

Each Q&A pair becomes:
```json
{
  "messages": [
    {"role": "system", "content": "You are a helpful e-commerce agent..."},
    {"role": "user", "content": "<question from FAQ>"},
    {"role": "assistant", "content": "<answer from FAQ>"}
  ]
}
```

In [ ]:
# Load system prompt
with open('../data/system_prompts.json') as f:
    system_prompts = json.load(f)

SYSTEM_PROMPT = system_prompts['ecommerce_agent']

def format_as_instruction(question, answer, system_prompt):
    """Convert a single Q&A pair into chat message format."""
    return {
        'messages': [
            {'role': 'system', 'content': system_prompt},
            {'role': 'user', 'content': question},
            {'role': 'assistant', 'content': answer}
        ]
    }

# Convert all FAQ data
instruction_data = []
for _, row in faq_df.iterrows():
    sample = format_as_instruction(row['question'], row['answer'], SYSTEM_PROMPT)
    instruction_data.append(sample)

# Save instruction data
instructions_path = Path('../data/ecommerce_instructions.jsonl')
with open(instructions_path, 'w') as f:
    for sample in instruction_data:
        f.write(json.dumps(sample) + '\n')

print(f'[OK] Converted {len(instruction_data)} FAQ pairs to instruction format')
print(f'  Saved to {instructions_path}')

## 4. Generate Multi-Turn Conversations

Single-turn Q&A is good, but real customer conversations have **multiple turns**.

We'll use our LLM API to generate synthetic multi-turn conversations grounded in the FAQ data.

In [ ]:
SCENARIOS = [
    {'topic': 'return_process', 'description': 'Customer wants to return a product. Walk through the full return process including policy details, requesting a return label, and refund timeline.', 'num_conversations': 10},
    {'topic': 'order_inquiry', 'description': 'Customer asks about their order status, shipping timeline, or tracking information. Include follow-up questions about delivery estimates.', 'num_conversations': 10},
    {'topic': 'shipping_questions', 'description': 'Customer has questions about shipping options, international shipping, changing shipping address, or shipping delays.', 'num_conversations': 10},
    {'topic': 'product_availability', 'description': 'Customer asks about product availability, stock status, pre-orders, or when out-of-stock items will be available again.', 'num_conversations': 8},
    {'topic': 'payment_and_pricing', 'description': 'Customer has questions about payment methods, promo codes, price matching, price adjustments, or billing issues.', 'num_conversations': 8},
    {'topic': 'account_and_general', 'description': 'Customer needs help with account creation, loyalty program, gift orders, newsletter, or general store policies.', 'num_conversations': 8},
    {'topic': 'complaints_and_issues', 'description': 'Customer received wrong item, damaged product, or has a complaint. Agent should be empathetic and offer solutions.', 'num_conversations': 6},
]

total_planned = sum(s['num_conversations'] for s in SCENARIOS)
print(f'Planned: {total_planned} conversations across {len(SCENARIOS)} scenarios')

In [ ]:
faq_context = '\n'.join(
    f'Q: {row["question"]}\nA: {row["answer"]}\n'
    for _, row in faq_df.iterrows()
)

def generate_conversation(scenario, faq_context, system_prompt):
    """Generate a single multi-turn conversation using the LLM API."""
    generation_prompt = f"""Generate a realistic multi-turn customer service conversation for an e-commerce store called ShopEasy.

SCENARIO: {scenario['description']}

REFERENCE FAQ DATA (use this to ensure accuracy):
{faq_context[:3000]}

RULES:
- Generate 3-6 turns (alternating user/assistant messages)
- The assistant should be concise, friendly, and professional
- Use realistic customer language (casual, sometimes frustrated)
- Include specific details (order numbers like ORD-XXXXX, product names, dates)
- The assistant's answers should be consistent with the FAQ data above
- Do NOT include any system message -- just the user/assistant turns

Return ONLY a valid JSON array of message objects:
[
  {{"role": "user", "content": "..."}},
  {{"role": "assistant", "content": "..."}},
  ...
]"""
    
    response = llm_client.chat.completions.create(
        model=LLM_MODEL,
        messages=[{'role': 'user', 'content': generation_prompt}],
        temperature=0.9, max_tokens=1000
    )
    
    content = response.choices[0].message.content.strip()
    if '```json' in content:
        content = content.split('```json')[1].split('```')[0].strip()
    elif '```' in content:
        content = content.split('```')[1].split('```')[0].strip()
    
    turns = json.loads(content)
    messages = [{'role': 'system', 'content': system_prompt}] + turns
    return {'messages': messages}

print('[OK] Conversation generator ready')

In [ ]:
# Generate all conversations
conversation_data = []
errors = 0

for scenario in SCENARIOS:
    topic = scenario['topic']
    print(f'\nGenerating {scenario["num_conversations"]} conversations for: {topic}')
    
    for i in range(scenario['num_conversations']):
        try:
            conv = generate_conversation(scenario, faq_context, SYSTEM_PROMPT)
            conversation_data.append(conv)
            print(f'  [{i+1}/{scenario["num_conversations"]}] {len(conv["messages"])-1} turns', end='\r')
        except Exception as e:
            errors += 1
            print(f'  [{i+1}] Error: {str(e)[:50]}')
    
    print(f'  Done: {topic} ({scenario["num_conversations"]} conversations)')

# Save conversations
conversations_path = Path('../data/ecommerce_conversations.jsonl')
with open(conversations_path, 'w') as f:
    for conv in conversation_data:
        f.write(json.dumps(conv) + '\n')

print(f'\n[OK] Generated {len(conversation_data)} conversations ({errors} errors)')
print(f'  Saved to {conversations_path}')

## 5. Create Train/Eval Split

In [ ]:
# Combine all data
all_data = instruction_data + conversation_data
random.seed(42)
random.shuffle(all_data)

# 90/10 split
split_idx = int(len(all_data) * 0.9)
train_data = all_data[:split_idx]
eval_data = all_data[split_idx:]

# Save
for path, data in [('../data/train_set.jsonl', train_data), ('../data/eval_set.jsonl', eval_data)]:
    with open(path, 'w') as f:
        for sample in data:
            f.write(json.dumps(sample) + '\n')

print(f'[OK] Dataset split')
print(f'  Total: {len(all_data)} | Train: {len(train_data)} | Eval: {len(eval_data)}')

In [ ]:
# Validate
issues = []
for i, sample in enumerate(train_data):
    msgs = sample.get('messages', [])
    if not msgs: issues.append(f'Sample {i}: Empty')
    elif msgs[0]['role'] != 'system': issues.append(f'Sample {i}: No system msg')
    elif msgs[-1]['role'] != 'assistant': issues.append(f'Sample {i}: No assistant msg')

print(f'Validation: {len(issues)} issues found')
for issue in issues[:3]: print(f'  - {issue}')

---
## Part B: Fine-Tuning with QLoRA
---

## 6. Load Model in 4-bit

In [ ]:
OUTPUT_DIR = Path('../results/ecommerce-ft-v1')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Load model in 4-bit
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type='nf4',
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID, quantization_config=bnb_config,
    device_map='auto', token=HF_TOKEN, trust_remote_code=True
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, token=HF_TOKEN, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
    tokenizer.pad_token_id = tokenizer.eos_token_id

print(f'[OK] Model loaded in 4-bit')
print(f'  VRAM used: {torch.cuda.memory_allocated() / (1024**3):.1f} GB')

## 7. Prepare Training Dataset

In [ ]:
# Convert to HuggingFace Dataset
def format_sample(sample):
    """Apply chat template to convert messages into training text."""
    text = tokenizer.apply_chat_template(
        sample['messages'], tokenize=False, add_generation_prompt=False
    )
    return {'text': text}

train_dataset = Dataset.from_list(train_data)
train_dataset = train_dataset.map(format_sample, remove_columns=train_dataset.column_names)

print(f'[OK] Training data: {len(train_dataset)} samples')
print(f'\nSample (first 200 chars):')
print(train_dataset[0]['text'][:200] + '...')

## 8. Configure LoRA

LoRA adds small trainable matrices to specific layers. Key parameters:
- **r (rank)**: Size of LoRA matrices. 16 is a good default.
- **lora_alpha**: Scaling factor. Usually 2x the rank.
- **target_modules**: Which layers to apply LoRA to. Attention layers work best.

In [ ]:
with open('../configs/lora_config.json') as f:
    lora_params = json.load(f)

lora_config = LoraConfig(
    r=lora_params['r'],                        # Rank: 16
    lora_alpha=lora_params['lora_alpha'],       # Alpha: 32 (2x rank)
    target_modules=lora_params['target_modules'],  # Attention layers
    lora_dropout=lora_params['lora_dropout'],   # Dropout: 0.05
    bias=lora_params['bias'],
    task_type=TaskType.CAUSAL_LM
)

model = get_peft_model(model, lora_config)

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total = sum(p.numel() for p in model.parameters())

print(f'[OK] LoRA applied (r={lora_config.r}, alpha={lora_config.lora_alpha})')
print(f'  Trainable: {trainable:,} / {total:,} ({trainable/total*100:.2f}%)')

## 9. Configure Training and Train

In [ ]:
with open('../configs/training_config.json') as f:
    train_params = json.load(f)

training_args = SFTConfig(
    output_dir=str(OUTPUT_DIR),
    num_train_epochs=train_params['num_train_epochs'],
    per_device_train_batch_size=train_params['per_device_train_batch_size'],
    gradient_accumulation_steps=train_params['gradient_accumulation_steps'],
    learning_rate=train_params['learning_rate'],
    warmup_steps=train_params['warmup_steps'],
    logging_steps=train_params['logging_steps'],
    save_steps=train_params['save_steps'],
    fp16=train_params['fp16'],
    max_seq_length=train_params['max_seq_length'],
    optim=train_params['optim'],
    dataset_text_field='text',
    report_to='none',
)

print(f'[OK] Training config:')
print(f'  Epochs: {training_args.num_train_epochs} | Batch: {training_args.per_device_train_batch_size} x {training_args.gradient_accumulation_steps} = {training_args.per_device_train_batch_size * training_args.gradient_accumulation_steps}')
print(f'  LR: {training_args.learning_rate} | Max seq: {training_args.max_seq_length}')

In [ ]:
trainer = SFTTrainer(
    model=model, args=training_args,
    train_dataset=train_dataset, processing_class=tokenizer,
)

print('Starting training...')
print('=' * 50)

start_time = time.time()
train_result = trainer.train()
training_time = time.time() - start_time

print('=' * 50)
print(f'[OK] Training complete!')
print(f'  Time: {training_time/60:.1f} min | Loss: {train_result.training_loss:.4f} | Steps: {train_result.global_step}')

In [ ]:
# Plot training loss
log_history = trainer.state.log_history
losses = [(log['step'], log['loss']) for log in log_history if 'loss' in log]

if losses:
    steps, loss_values = zip(*losses)
    plt.figure(figsize=(10, 4))
    plt.plot(steps, loss_values, 'b-', linewidth=1.5)
    plt.xlabel('Training Step')
    plt.ylabel('Loss')
    plt.title('Training Loss Curve')
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()

## 10. Save Adapter

In [ ]:
adapter_path = OUTPUT_DIR / 'adapter'
model.save_pretrained(str(adapter_path))
tokenizer.save_pretrained(str(adapter_path))

adapter_size = sum(f.stat().st_size for f in adapter_path.rglob('*') if f.is_file()) / (1024 * 1024)

print(f'[OK] Adapter saved to {adapter_path}')
print(f'  Adapter size: {adapter_size:.1f} MB (vs ~2.5 GB full model)')

## 11. Quick Test: Before vs After

In [ ]:
def generate_response(model, tokenizer, query, system_prompt=None, max_new_tokens=256):
    """Generate a response from the model."""
    if system_prompt is None:
        system_prompt = 'You are a helpful e-commerce customer service agent.'
    messages = [
        {'role': 'system', 'content': system_prompt},
        {'role': 'user', 'content': query}
    ]
    input_ids = tokenizer.apply_chat_template(
        messages, return_tensors='pt', add_generation_prompt=True
    ).to(model.device)
    with torch.no_grad():
        outputs = model.generate(
            input_ids, max_new_tokens=max_new_tokens,
            temperature=0.7, do_sample=True, pad_token_id=tokenizer.pad_token_id
        )
    return tokenizer.decode(outputs[0][input_ids.shape[1]:], skip_special_tokens=True).strip()

# Load baseline responses from Notebook 01
with open('../data/baseline_responses.json') as f:
    baseline_data = json.load(f)

print('BEFORE vs AFTER Fine-Tuning')
print('=' * 70)

for item in baseline_data:
    query = item['query']
    baseline = item['baseline_response']
    expected = item['expected']
    ft_response = generate_response(model, tokenizer, query)
    
    print(f'\nQ: {query}')
    print(f'\n  BASE MODEL:  {baseline[:150]}...' if len(baseline) > 150 else f'\n  BASE MODEL:  {baseline}')
    print(f'\n  FINE-TUNED:  {ft_response[:150]}...' if len(ft_response) > 150 else f'\n  FINE-TUNED:  {ft_response}')
    print(f'\n  EXPECTED:    {expected[:150]}...' if len(expected) > 150 else f'\n  EXPECTED:    {expected}')
    print('-' * 70)

## 12. Summary

### Data Preparation
| Dataset | Samples |
|---------|--------|
| FAQ Instructions | 79 |
| Synthetic Conversations | ~60 |
| Train Split | ~125 (90%) |
| Eval Split | ~14 (10%) |

### Fine-Tuning
| Aspect | Value |
|--------|-------|
| Model | Llama 3.2 1B Instruct (4-bit) |
| Method | QLoRA (r=16, alpha=32) |
| Trainable | ~0.5% of parameters |
| Adapter size | ~10-50 MB |

**Next:** Evaluate the fine-tuned model with ROUGE, semantic similarity, and LLM-as-judge.